# Gages Activity Geodatabase creation

In [34]:
import pandas as pd
import geopandas as gpd
import shapely
import pyogrio
from pathlib import Path
from osgeo import gdal
import shutil
from osgeo import gdal


## Source for original gages csv I created

In [35]:
import pandas as pd
from pathlib import Path

csv_path = Path(
    "/Users/sophiameyers/Documents/GitHub/USGS-Gages-Research/data/intermediate/gages_activity_with_api_locations_recovered.csv"
)

df = pd.read_csv(csv_path, low_memory=False)

print("Rows and columns:", df.shape)
print("Column names:")
print(df.columns.tolist())

print("First 5 rows:")
display(df.head())

print("State names:")
display(df["state_name"].value_counts().head(20))

Rows and columns: (23242, 13)

Column names:
['site', 'n_years_active', 'earliest_active_year', 'which_years_active', 'any_gaps', 'gap_years', 'api_id', 'station_name', 'state_name', 'latitude', 'longitude', 'location_found', 'coordinates_available']

First 5 rows:


,site,n_years_active,earliest_active_year,which_years_active,any_gaps,gap_years,api_id,station_name,state_name,latitude,longitude,location_found,coordinates_available
0,7010000,159,1863,1863;1864;1865;1866;1867;1868;1869;1870;1871;1...,False,NaN,USGS-07010000,"Mississippi River at St. Louis, MO",Missouri,38.629000,-90.179778,True,True
1,5420500,148,1874,1874;1875;1876;1877;1878;1879;1880;1881;1882;1...,False,NaN,USGS-05420500,"Mississippi River at Clinton, IA",Iowa,41.780586,-90.252073,True,True
2,3568000,134,1875,1875;1876;1877;1878;1879;1880;1881;1882;1883;1...,False,NaN,USGS-03568000,"TENNESSEE RIVER AT CHATTANOOGA, TN",Tennessee,35.086776,-85.278430,True,True
3,3193000,144,1878,1878;1879;1880;1881;1882;1883;1884;1885;1886;1...,False,NaN,USGS-03193000,"KANAWHA RIVER AT KANAWHA FALLS, WV",West Virginia,38.138163,-81.214274,True,True
4,5474500,143,1879,1879;1880;1881;1882;1883;1884;1885;1886;1887;1...,False,NaN,USGS-05474500,"Mississippi River at Keokuk, IA",Iowa,40.393655,-91.374318,True,True



State names:


state_name
California        2327
Colorado          1160
Florida            961
Washington         941
Texas              922
Montana            710
Utah               698
Idaho              693
Oregon             669
New York           652
Wisconsin          645
Pennsylvania       624
Wyoming            601
North Carolina     540
Ohio               499
Arizona            488
Georgia            481
Alaska             474
Tennessee          431
Michigan           421
Name: count, dtype: int64

## Checking fields

In [36]:
required_columns = [
    "site",
    "state_name",
    "latitude",
    "longitude",
    "earliest_active_year",
    "which_years_active"
]

missing = [col for col in required_columns if col not in df.columns]

if missing:
    print("Missing required columns:", missing)
else:
    print("All required columns are present.")

    print("Missing coordinates:")
    print(df[["latitude", "longitude"]].isna().sum())

    print("Earliest active years:")
    print(df["earliest_active_year"].describe())

All required columns are present.

Missing coordinates:
latitude     3
longitude    3
dtype: int64

Earliest active years:
count    23242.000000
mean      1964.992987
std         30.864613
min       1863.000000
25%       1941.000000
50%       1965.000000
75%       1990.000000
max       2021.000000
Name: earliest_active_year, dtype: float64


## Check the 10 states and activity-year format

In [37]:
target_states = [
    "Nebraska",
    "Iowa",
    "Missouri",
    "Arkansas",
    "Oklahoma",
    "Mississippi",
    "Kentucky",
    "West Virginia",
    "Alabama",
    "Tennessee",
]

print("Requested states found:")
print(
    df.loc[df["state_name"].isin(target_states), "state_name"]
      .value_counts()
      .reindex(target_states, fill_value=0)
)

for i, value in df["which_years_active"].dropna().head(5).items():
    years = [
        int(year.strip())
        for year in str(value).split(";")
        if year.strip().isdigit()
    ]
    print(f"Row {i}: first years={years[:5]}, last years={years[-3:]}")

print("Rows missing coordinates:")
print(
    df.loc[
        df["latitude"].isna() | df["longitude"].isna(),
        ["site", "station_name", "state_name", "latitude", "longitude"]
    ]
)

Requested states found:
state_name
Nebraska         319
Iowa             252
Missouri         397
Arkansas         260
Oklahoma         339
Mississippi      215
Kentucky         321
West Virginia    288
Alabama          361
Tennessee        431
Name: count, dtype: int64
Row 0: first years=[1863, 1864, 1865, 1866, 1867], last years=[2019, 2020, 2021]
Row 1: first years=[1874, 1875, 1876, 1877, 1878], last years=[2019, 2020, 2021]
Row 2: first years=[1875, 1876, 1877, 1878, 1879], last years=[2006, 2007, 2008]
Row 3: first years=[1878, 1879, 1880, 1881, 1882], last years=[2019, 2020, 2021]
Row 4: first years=[1879, 1880, 1881, 1882, 1883], last years=[2019, 2020, 2021]

Rows missing coordinates:
                  site                                     station_name  \
10535         11516100  GREEN SPRINGS POWERPLANT DIVER NR ASHLAND,OREG.   
22351  421247071470201                    BRIDGE-RT20W-QUINSIGAMOND, MA   
22353  422108071052501                          BRIDGE-RT2AE-BOSTON, MA 

# Intentionally skipped.
# The final projected geodatabases are created in Cell 14.

In [38]:

output_folder = (
    Path.home()
    / "Documents"
    / "GitHub"
    / "USGS-Gages-Research"
    / "data"
    / "processed"
    / "Gages_State_GDBs"
)
output_folder.mkdir(parents=True, exist_ok=True)

target_states = [
    "Nebraska",
    "Iowa",
    "Missouri",
    "Arkansas",
    "Oklahoma",
    "Mississippi",
    "Kentucky",
    "West Virginia",
    "Alabama",
    "Tennessee",
]

snapshot_years = list(range(1860, 2030, 10))

print("Output folder:", output_folder)
print("Target states:", len(target_states))
print(target_states)
print("Snapshot years:", snapshot_years)

Output folder: /Users/sophiameyers/Documents/GitHub/USGS-Gages-Research/data/processed/Gages_State_GDBs
Target states: 10
['Nebraska', 'Iowa', 'Missouri', 'Arkansas', 'Oklahoma', 'Mississippi', 'Kentucky', 'West Virginia', 'Alabama', 'Tennessee']
Snapshot years: [1860, 1870, 1880, 1890, 1900, 1910, 1920, 1930, 1940, 1950, 1960, 1970, 1980, 1990, 2000, 2010, 2020]


# Intentionally skipped.
# The final projected geodatabases are created in Cell 14.

## Check outputs

In [39]:

for state in target_states:
    gdb_path = output_folder / f"{state.replace(' ', '_')}.gdb"

    if not gdb_path.exists():
        print(f"MISSING: {gdb_path.name}")
        continue

    layers = pyogrio.list_layers(gdb_path)
    print(f"{state}: {len(layers)} layers")


Nebraska: 13 layers
Iowa: 15 layers
Missouri: 16 layers
Arkansas: 12 layers
Oklahoma: 12 layers
MISSING: Mississippi.gdb
Kentucky: 14 layers
West Virginia: 15 layers
Alabama: 13 layers
Tennessee: 15 layers


## Change to NAD 83 with a UTM in meters

In [40]:
output_folder = (
    Path.home()
    / "Documents"
    / "GitHub"
    / "USGS-Gages-Research"
    / "data"
    / "processed"
    / "Gages_State_GDBs"
)
output_folder.mkdir(parents=True, exist_ok=True)

SOURCE_CRS = "EPSG:4326"

state_crs = {
    "Nebraska": "EPSG:26914",
    "Iowa": "EPSG:26915",
    "Missouri": "EPSG:26915",
    "Arkansas": "EPSG:26915",
    "Oklahoma": "EPSG:26914",
    "Mississippi": "EPSG:26915",
    "Kentucky": "EPSG:26916",
    "West Virginia": "EPSG:26917",
    "Alabama": "EPSG:26916",
    "Tennessee": "EPSG:26916",
}

target_states = list(state_crs.keys())
snapshot_years = list(range(1860, 2030, 10))

work = df.copy()

work["latitude"] = pd.to_numeric(
    work["latitude"], errors="coerce"
)
work["longitude"] = pd.to_numeric(
    work["longitude"], errors="coerce"
)
work["earliest_active_year"] = pd.to_numeric(
    work["earliest_active_year"], errors="coerce"
)

def parse_active_years(value):
    if pd.isna(value):
        return set()

    return {
        int(year.strip())
        for year in str(value).split(";")
        if year.strip().isdigit()
    }

work["_active_years"] = work["which_years_active"].apply(
    parse_active_years
)

valid_coordinates = (
    work["latitude"].between(-90, 90)
    & work["longitude"].between(-180, 180)
)

work = work[
    work["state_name"].isin(target_states)
    & valid_coordinates
].copy()

print("Source gages with valid coordinates:", len(work))
print("States represented:")
print(
    work["state_name"]
    .value_counts()
    .reindex(target_states, fill_value=0)
)

gdb_paths = {
    state: output_folder / f"{state.replace(' ', '_')}.gdb"
    for state in target_states
}

for path in gdb_paths.values():
    if path.exists():
        print("Removing existing output:", path.name)
        shutil.rmtree(path)

layer_counts = []

for state, target_crs in state_crs.items():

    state_df = work[work["state_name"] == state].copy()
    gdb_path = gdb_paths[state]

    print(f"\nCreating {gdb_path.name} in {target_crs}")
    print("Source gages:", len(state_df))

    for i, year in enumerate(snapshot_years):

        eligible = state_df[
            state_df["earliest_active_year"].notna()
            & (state_df["earliest_active_year"] <= year)
        ].copy()

        eligible["activity_status"] = eligible["_active_years"].apply(
            lambda years: "Active" if year in years else "Inactive"
        )
        eligible["snapshot_year"] = year
        eligible = eligible.drop(columns=["_active_years"])

        gdf = gpd.GeoDataFrame(
            eligible,
            geometry=gpd.points_from_xy(
                eligible["longitude"],
                eligible["latitude"]
            ),
            crs=SOURCE_CRS,
        )

        gdf = gdf.to_crs(target_crs)

        for col in ["site", "api_id"]:
            if col in gdf.columns:
                gdf[col] = gdf[col].astype("string")

        layer_name = f"{state.replace(' ', '_')}_Gages_{year}"

        gdf.to_file(
            gdb_path,
            layer=layer_name,
            driver="OpenFileGDB",
            engine="pyogrio",
            geometry_type="Point",
            mode="w" if i == 0 else "a",
        )

        layer_counts.append({
            "state": state,
            "crs": target_crs,
            "year": year,
            "layer": layer_name,
            "feature_count": len(gdf),
            "active_count": int(
                (gdf["activity_status"] == "Active").sum()
            ),
            "inactive_count": int(
                (gdf["activity_status"] == "Inactive").sum()
            ),
        })

    print(f"Finished {state}")

counts_df = pd.DataFrame(layer_counts)

print("Export finished.")
print("Geodatabases created:", len(gdb_paths))
print("Layers attempted:", len(counts_df))
display(counts_df.head(20))

Source gages with valid coordinates: 3183
States represented:
state_name
Nebraska         319
Iowa             252
Missouri         397
Arkansas         260
Oklahoma         339
Mississippi      215
Kentucky         321
West Virginia    288
Alabama          361
Tennessee        431
Name: count, dtype: int64
Removing existing output: Nebraska.gdb
Removing existing output: Iowa.gdb
Removing existing output: Missouri.gdb
Removing existing output: Arkansas.gdb
Removing existing output: Oklahoma.gdb
Removing existing output: Kentucky.gdb
Removing existing output: West_Virginia.gdb
Removing existing output: Alabama.gdb
Removing existing output: Tennessee.gdb

Creating Nebraska.gdb in EPSG:26914
Source gages: 319


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field n_years_active of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field earliest_active_year of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field snapshot_year of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field n_years_active of type Integer64 will be written as a Float64. To get Integer64, use laye

Finished Nebraska

Creating Iowa.gdb in EPSG:26915
Source gages: 252
Finished Iowa

Creating Missouri.gdb in EPSG:26915
Source gages: 397


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field n_years_active of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field earliest_active_year of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field snapshot_year of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field n_years_active of type Integer64 will be written as a Float64. To get Integer64, use laye

Finished Missouri

Creating Arkansas.gdb in EPSG:26915
Source gages: 260
Finished Arkansas

Creating Oklahoma.gdb in EPSG:26914
Source gages: 339


...

Finished Kentucky

Creating West_Virginia.gdb in EPSG:26917
Source gages: 288
Finished West Virginia

Creating Alabama.gdb in EPSG:26916
Source gages: 361


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field n_years_active of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field earliest_active_year of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field snapshot_year of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field n_years_active of type Integer64 will be written as a Float64. To get Integer64, use laye

Finished Alabama

Creating Tennessee.gdb in EPSG:26916
Source gages: 431
Finished Tennessee

Export finished.
Geodatabases created: 10
Layers attempted: 170


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field n_years_active of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field earliest_active_year of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field snapshot_year of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field n_years_active of type Integer64 will be written as a Float64. To get Integer64, use laye

,state,crs,year,layer,feature_count,active_count,inactive_count
0,Nebraska,EPSG:26914,1860,Nebraska_Gages_1860,0,0,0
1,Nebraska,EPSG:26914,1870,Nebraska_Gages_1870,0,0,0
2,Nebraska,EPSG:26914,1880,Nebraska_Gages_1880,0,0,0
3,Nebraska,EPSG:26914,1890,Nebraska_Gages_1890,0,0,0
4,Nebraska,EPSG:26914,1900,Nebraska_Gages_1900,1,0,1
5,Nebraska,EPSG:26914,1910,Nebraska_Gages_1910,1,0,1
6,Nebraska,EPSG:26914,1920,Nebraska_Gages_1920,3,0,3
7,Nebraska,EPSG:26914,1930,Nebraska_Gages_1930,17,15,2
8,Nebraska,EPSG:26914,1940,Nebraska_Gages_1940,63,56,7
9,Nebraska,EPSG:26914,1950,Nebraska_Gages_1950,150,138,12


## check output

In [41]:
for state, gdb_path in gdb_paths.items():
    layers = pyogrio.list_layers(gdb_path)

    test_layer = f"{state.replace(' ', '_')}_Gages_2020"

    check = gpd.read_file(
        gdb_path,
        layer=test_layer,
        engine="pyogrio",
    )

    print(
        f"{state}: {len(layers)} layers | "
        f"CRS: {check.crs} | "
        f"units: {check.crs.axis_info[0].unit_name if check.crs else 'unknown'}"
    )

Nebraska: 17 layers | CRS: EPSG:26914 | units: metre
Iowa: 17 layers | CRS: EPSG:26915 | units: metre
Missouri: 17 layers | CRS: EPSG:26915 | units: metre
Arkansas: 17 layers | CRS: EPSG:26915 | units: metre
Oklahoma: 17 layers | CRS: EPSG:26914 | units: metre
Mississippi: 17 layers | CRS: EPSG:26915 | units: metre
Kentucky: 17 layers | CRS: EPSG:26916 | units: metre
West Virginia: 17 layers | CRS: EPSG:26917 | units: metre
Alabama: 17 layers | CRS: EPSG:26916 | units: metre
Tennessee: 17 layers | CRS: EPSG:26916 | units: metre


### All created 17 files accounting for the earliest gage, but not every state starting gaging at the same point so we have to remove layers created with empty attribute tables.

In [42]:
removed_layers = []
kept_layers = []
errors = []

for state, gdb_path in gdb_paths.items():
    print(f"\nChecking {state}...")

    ds = gdal.OpenEx(
        str(gdb_path),
        gdal.OF_VECTOR | gdal.OF_UPDATE
    )

    if ds is None:
        print(f"  ERROR: Could not open {gdb_path}")
        errors.append((state, "OPEN", "Could not open geodatabase"))
        continue

    layer_names = [
        ds.GetLayerByIndex(i).GetName()
        for i in range(ds.GetLayerCount())
    ]

    for layer_name in layer_names:
        layer = ds.GetLayerByName(layer_name)

        if layer is None:
            errors.append((state, layer_name, "Could not access layer"))
            continue

        feature_count = layer.GetFeatureCount()

        if feature_count == 0:
            layer_index = ds.GetLayerCount()
            for i in range(ds.GetLayerCount()):
                if ds.GetLayerByIndex(i).GetName() == layer_name:
                    layer_index = i
                    break

            if layer_index < ds.GetLayerCount():
                result = ds.DeleteLayer(layer_index)

                if result == 0:
                    removed_layers.append((state, layer_name))
                    print(f"  Removed {layer_name}")
                else:
                    errors.append(
                        (state, layer_name, "DeleteLayer failed")
                    )
        else:
            kept_layers.append((state, layer_name, feature_count))

    ds = None 

print("Empty layers removed:", len(removed_layers))
print("Non-empty layers kept:", len(kept_layers))
print("Errors:", len(errors))

if errors:
    print("Errors to review:")
    for error in errors:
        print(error)


Checking Nebraska...
  Removed Nebraska_Gages_1860
  Removed Nebraska_Gages_1870
  Removed Nebraska_Gages_1880
  Removed Nebraska_Gages_1890

Checking Iowa...
  Removed Iowa_Gages_1860
  Removed Iowa_Gages_1870

Checking Missouri...
  Removed Missouri_Gages_1860

Checking Arkansas...
  Removed Arkansas_Gages_1860
  Removed Arkansas_Gages_1870
  Removed Arkansas_Gages_1880
  Removed Arkansas_Gages_1890
  Removed Arkansas_Gages_1900

Checking Oklahoma...
  Removed Oklahoma_Gages_1860
  Removed Oklahoma_Gages_1870
  Removed Oklahoma_Gages_1880
  Removed Oklahoma_Gages_1890
  Removed Oklahoma_Gages_1900

Checking Mississippi...
  Removed Mississippi_Gages_1860
  Removed Mississippi_Gages_1870
  Removed Mississippi_Gages_1880
  Removed Mississippi_Gages_1890

Checking Kentucky...
  Removed Kentucky_Gages_1860
  Removed Kentucky_Gages_1870
  Removed Kentucky_Gages_1880

Checking West Virginia...
  Removed West_Virginia_Gages_1860
  Removed West_Virginia_Gages_1870

Checking Alabama...
  Rem

In [43]:
for state, gdb_path in gdb_paths.items():
    if not gdb_path.exists():
        print(f"MISSING: {state}")
        continue

    layers = pyogrio.list_layers(gdb_path)
    print(f"\n{state}.gdb — {len(layers)} layers")

    for layer_info in layers[:3]:
        print("  ", layer_info[0])

print("Total geodatabases:", len(gdb_paths))


Nebraska.gdb — 13 layers
   Nebraska_Gages_1900
   Nebraska_Gages_1910
   Nebraska_Gages_1920

Iowa.gdb — 15 layers
   Iowa_Gages_1880
   Iowa_Gages_1890
   Iowa_Gages_1900

Missouri.gdb — 16 layers
   Missouri_Gages_1870
   Missouri_Gages_1880
   Missouri_Gages_1890

Arkansas.gdb — 12 layers
   Arkansas_Gages_1910
   Arkansas_Gages_1920
   Arkansas_Gages_1930

Oklahoma.gdb — 12 layers
   Oklahoma_Gages_1910
   Oklahoma_Gages_1920
   Oklahoma_Gages_1930

Mississippi.gdb — 13 layers
   Mississippi_Gages_1900
   Mississippi_Gages_1910
   Mississippi_Gages_1920

Kentucky.gdb — 14 layers
   Kentucky_Gages_1890
   Kentucky_Gages_1900
   Kentucky_Gages_1910

West Virginia.gdb — 15 layers
   West_Virginia_Gages_1880
   West_Virginia_Gages_1890
   West_Virginia_Gages_1900

Alabama.gdb — 13 layers
   Alabama_Gages_1900
   Alabama_Gages_1910
   Alabama_Gages_1920

Tennessee.gdb — 15 layers
   Tennessee_Gages_1880
   Tennessee_Gages_1890
   Tennessee_Gages_1900

Total geodatabases: 10
